# 🩺 第三十三天 · 赛题十三零样本验证（最小闭环）

**今日目标（约 1.5 小时）**：跑通「古籍原文 → DeepSeek 零样本抽取 → 人工打分」完整闭环，拿到你赛题十三的**第一个基线分数**。

**流程**：
1. 20 条古籍句子（已备好）+ 草稿标准答案（**你终审**，医学裁决权在你）
2. 批量调用 DeepSeek 抽取 → 存结果
3. 打分脚本：对比模型输出 vs 标准答案 → 精确率 / 召回率 / F1

> ⚠️ 依赖 D32：`api_key.txt` 必须已建好、基础调用已跑通，否则先补 D32。

## 第 0 步 · 先终审标准答案（最重要！）

下面 20 条古籍句子的「标准答案」是我**草拟**的。按 D23 立的铁律：**医学裁决权在你**——请逐条核对，尤其这几类我拿不准的：
- 「太阳病」「少阴病」是**疾病**还是**证候**？
- 「阳虚」「阴虚」是**病机**还是**证候**？

**改法**：直接在下面 code cell 的 `DATA` 里改 `type` 或增删实体，改完保存。这是你的 gold（标准答案），模型分就按它算。

In [4]:
import json

# 20 条古籍 + 草稿标准答案（请你终审后保存）
DATA = [
 {"id":"1","s":"太阳病，头痛发热，汗出恶风，桂枝汤主之。",
  "gold":[("太阳病","疾病"),("头痛","症状"),("发热","症状"),("汗出","症状"),("恶风","症状"),("桂枝汤","方剂")]},
 {"id":"2","s":"少阴病，心中烦，不得卧，黄连阿胶汤主之。",
  "gold":[("少阴病","疾病"),("心中烦","症状"),("不得卧","症状"),("黄连阿胶汤","方剂")]},
 {"id":"3","s":"伤寒表不解，干呕发热而咳，小青龙汤主之。",
  "gold":[("干呕","症状"),("发热","症状"),("咳","症状"),("小青龙汤","方剂")]},
 {"id":"4","s":"太阴之为病，腹满而吐，食不下，自利益甚。",
  "gold":[("腹满","症状"),("吐","症状"),("食不下","症状"),("自利","症状")]},
 {"id":"5","s":"发汗后，身疼痛，脉沉迟者，桂枝加芍药汤主之。",
  "gold":[("发汗","治法"),("身疼痛","症状"),("脉沉迟","症状"), ("桂枝加芍药汤","方剂")]},
 {"id":"6","s":"阳明病，胃家实是也。",
  "gold":[("阳明病","疾病"),("胃家实","病机")]},
 {"id":"7","s":"病痰饮者，当以温药和之。",
  "gold":[("痰饮","疾病"),("温药和之","治法")]},
 {"id":"8","s":"脉浮，发热，渴欲饮水，小便不利者，猪苓汤主之。",
  "gold":[("脉浮","症状"), ("发热","症状"),("渴欲饮水","症状"),("小便不利","症状"),("猪苓汤","方剂")]},
 {"id":"9","s":"人参，味甘微寒，主补五脏，安精神。",
  "gold":[("人参","中药"),("补五脏","治法"),("安精神","治法")]},
 {"id":"10","s":"附子，生用则发散，熟用则峻补。",
  "gold":[("附子","中药"),("生用","炮制"),("熟用","炮制"), ("发散","治法"), ("峻补","治法")]},
 {"id":"11","s":"甘草，炙则温中，生则泻火。",
  "gold":[("甘草","中药"),("炙","炮制"),("生","炮制"),("温中","治法"),("泻火","治法")]},
 {"id":"12","s":"麻黄，去节，先煮，去上沫。",
  "gold":[("麻黄","中药"),("去节","炮制"),("先煮","炮制"),("去上沫","炮制")]},
 {"id":"13","s":"足阳明胃经，起于鼻，交頞中。",
  "gold":[("足阳明胃经","经络")]},
 {"id":"14","s":"灸足三里，可强身健体。",
  "gold":[("灸","治法"),("足三里","穴位")]},
 {"id":"15","s":"阳虚则外寒，阴虚则内热。",
  "gold":[("阳虚","病机"),("外寒","病机"),("阴虚","病机"),("内热","病机")]},
 {"id":"16","s":"气虚者，宜补气。",
  "gold":[("气虚","证候"),("补气","治法")]},
 {"id":"17","s":"风寒束表，肺气不宣，宜发汗解表。",
  "gold":[("风寒束表","证候"),("肺气不宣","病机"),("发汗解表","治法")]},
 {"id":"18","s":"心血不足，心神失养，致心悸失眠。",
  "gold":[("心血不足","病机"),("心悸","症状"),("失眠","症状")]},
 {"id":"19","s":"肝阳上亢，头晕目眩，宜平肝潜阳。",
  "gold":[("肝阳上亢","证候"),("头晕","症状"),("目眩","症状"),("平肝潜阳","治法")]},
 {"id":"20","s":"四物汤，由熟地黄、当归、白芍、川芎组成。",
  "gold":[("四物汤","方剂"),("熟地黄","中药"),("当归","中药"),("白芍","中药"),("川芎","中药")]},
]
print("已载入", len(DATA), "条；请先核对 gold 再往下跑")

已载入 20 条；请先核对 gold 再往下跑


## 第 1 步 · 零样本 vs 少样本（30 秒）

- **零样本**：不举例，只给 10 类定义 + 输出格式，让模型直接抽——**今天做的**；
- **少样本（few-shot）**：在 prompt 里塞 3–5 个「原文→正确结果」的示例——模型照葫芦画瓢，通常更准，**D38 再升级**。

先跑零样本拿基线分，才知道少样本能提升多少。

In [5]:
from openai import OpenAI
import time

with open("api_key.txt", encoding="utf-8") as f:
    api_key = f.read().strip()
client = OpenAI(api_key=api_key, base_url="https://api.deepseek.com")

PROMPT = """你是一名中医古籍实体识别专家。从下面古文原文中识别 10 类实体：
- 疾病：病名（如消渴、痰饮）
- 证候：辨证分型（如风寒束表、气虚）
- 症状：临床表现（如头痛、发热、恶风）
- 中药：药材名（如人参、桂枝、麻黄）
- 方剂：方名（如桂枝汤、四物汤）
- 治法：治疗原则/方法（如发汗、补气、平肝潜阳）
- 经络：经脉名（如足阳明胃经）
- 穴位：腧穴名（如足三里）
- 炮制：药材加工方法（如炙、炒、去节、生用）
- 病机：病理机制（如胃家实、阴虚内热）

要求：
1. 只输出 JSON，不要任何其他文字；
2. 格式：{"entities": [{"text": "实体原文", "type": "类别"}]}
3. text 必须一字不差来自原文；不确定的类别宁可不标。

古文：
"""

results = {}
for item in DATA:
    resp = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": PROMPT + item["s"]}],
        response_format={"type": "json_object"},
    )
    results[item["id"]] = json.loads(resp.choices[0].message.content)
    print(item["id"], "完成:", results[item["id"]].get("entities"))
    time.sleep(0.5)   # 限速保护，避免触发频率限制

json.dump(results, open("d33_results.json", "w"), ensure_ascii=False, indent=1)
print("\n结果已保存到 d33_results.json")

1 完成: [{'text': '太阳病', 'type': '疾病'}, {'text': '头痛', 'type': '症状'}, {'text': '发热', 'type': '症状'}, {'text': '汗出恶风', 'type': '症状'}, {'text': '桂枝汤', 'type': '方剂'}]
2 完成: [{'text': '少阴病', 'type': '疾病'}, {'text': '心中烦', 'type': '症状'}, {'text': '不得卧', 'type': '症状'}, {'text': '黄连阿胶汤', 'type': '方剂'}]
3 完成: [{'text': '伤寒', 'type': '疾病'}, {'text': '表不解', 'type': '证候'}, {'text': '干呕', 'type': '症状'}, {'text': '发热', 'type': '症状'}, {'text': '咳', 'type': '症状'}, {'text': '小青龙汤', 'type': '方剂'}]
4 完成: [{'text': '太阴之为病', 'type': '疾病'}, {'text': '腹满', 'type': '症状'}, {'text': '吐', 'type': '症状'}, {'text': '食不下', 'type': '症状'}, {'text': '自利', 'type': '症状'}]
5 完成: [{'text': '发汗', 'type': '治法'}, {'text': '身疼痛', 'type': '症状'}, {'text': '脉沉迟', 'type': '症状'}, {'text': '桂枝加芍药汤', 'type': '方剂'}]
6 完成: [{'text': '阳明病', 'type': '疾病'}, {'text': '胃家实', 'type': '病机'}]
7 完成: [{'text': '痰饮', 'type': '疾病'}, {'text': '温药', 'type': '治法'}]
8 完成: [{'text': '脉浮', 'type': '症状'}, {'text': '发热', 'type': '症状'}, {'text': '渴欲饮水', 'typ

In [6]:
# 打分：对比模型输出 vs 你的标准答案
def score(gold, pred):
    g = set((e[0], e[1]) for e in gold)
    p = set((e["text"], e["type"]) for e in pred.get("entities", []))
    tp = len(g & p)
    prec = tp / len(p) if p else 0
    rec  = tp / len(g) if g else 0
    f1 = 2*prec*rec/(prec+rec) if (prec+rec) else 0
    return prec, rec, f1, tp, len(g), len(p)

tot_tp = tot_g = tot_p = 0
print(f'{"id":<3} {"精确率":<8} {"召回率":<8} {"F1":<8} {"漏标(召回漏)":<12}')
for item in DATA:
    prec, rec, f1, tp, g, p = score(item["gold"], results[item["id"]])
    tot_tp += tp; tot_g += g; tot_p += p
    print(f'{item["id"]:<3} {prec:<8.2%} {rec:<8.2%} {f1:<8.2%} {g-tp}')
print("-"*45)
P = tot_tp/tot_p if tot_p else 0
R = tot_tp/tot_g if tot_g else 0
F = 2*P*R/(P+R) if (P+R) else 0
print(f'总体精确率 {P:.2%} | 召回率 {R:.2%} | F1 {F:.2%}')

id  精确率      召回率      F1       漏标(召回漏)     
1   80.00%   66.67%   72.73%   2
2   100.00%  100.00%  100.00%  0
3   66.67%   100.00%  80.00%   0
4   80.00%   100.00%  88.89%   0
5   100.00%  100.00%  100.00%  0
6   100.00%  100.00%  100.00%  0
7   50.00%   50.00%   50.00%   1
8   100.00%  100.00%  100.00%  0
9   50.00%   33.33%   40.00%   2
10  100.00%  100.00%  100.00%  0
11  100.00%  100.00%  100.00%  0
12  100.00%  50.00%   66.67%   2
13  100.00%  100.00%  100.00%  0
14  100.00%  50.00%   66.67%   1
15  0.00%    0.00%    0.00%    4
16  100.00%  100.00%  100.00%  0
17  100.00%  100.00%  100.00%  0
18  50.00%   66.67%   57.14%   1
19  66.67%   50.00%   57.14%   2
20  100.00%  100.00%  100.00%  0
---------------------------------------------
总体精确率 83.82% | 召回率 79.17% | F1 81.43%


## 第 2 步 · 观察（写这里）

看总体 F1（这就是你赛题十三的**零样本基线分**）：
1. 精确率和召回率哪个更高？说明模型是「乱报」还是「漏抽」？ 精准率高，说明是在漏抽
2. 哪类实体最容易抽错/漏抽（疾病 vs 症状 vs 炮制…）？ 主要是治法遗漏，很多其实没漏也没错，但是切分标准不一样，比如“头晕目眩”我的答案里是算作两个症状，但是大模型只算了一个
3. 想提升，你的第一招是什么？在提示词中给出更加具体的例子

## ✅ D33 完成标准

- [ ] gold 已逐条终审（尤其太阳病/阳虚的类别）
- [ ] 批量抽取跑完，d33_results.json 生成
- [ ] 打分脚本跑通，拿到 F1 基线
- [ ] 观察写完
- [ ] 保存（Cmd + S）

> 完成后喊我验收。**D34 预告**：简历 v3 + 报名 CHIP（若还没报）+ 赛题13 官方样例数据下标约定确认。